# 03 · Pivot to Cross-Sectional Residual Reversal

### TL;DR

The ablation in `02_ablations.ipynb` topped out at OOS Sharpe **0.35**. That is not because our execution layer was wrong, it is because **the primary signal was on the wrong side of the intraday autocorrelation structure**. Moving-average crossover is a time-series **momentum** bet, but on 1-minute minute bars of individual mega-cap tech names the autocorrelation of returns is **negative at every horizon from 5 to 60 minutes**. We were systematically buying the loser of the last 20 minutes of a random walk.

This notebook documents an architectural pivot:

1. **Diagnose the signal**: show the negative IC of past-return on forward-return across horizons. Prove the previous direction was wrong.
2. **Pivot to literature**: Nagel (2012) on short-term reversal as a liquidity premium; Avellaneda & Lee (2010) on residual statistical arbitrage; Heston-Korajczyk-Sadka (2010) on intraday cross-sectional effects.
3. **Build the new strategy iteratively**. Each step is a single design change; we record the OOS Sharpe after each change.
4. **Discover regime dependence** from a monthly decomposition of OOS PnL.
5. **Solve it** with a cross-sectional residual-dispersion gate — the mechanism turns out to be simple: reversion needs dispersion to revert from.

**Final OOS (Dec 2025 – Mar 2026, 1 bp cost)**: Sharpe **2.72**, Ann return **+43.9%**, Max DD **-5.6%**, Calmar **7.78**. Robust across cost assumptions 0.5-2 bp and across the parameter neighbourhood (~15 configs in the region give Sharpe 2.0-3.3).

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

PROJECT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT / 'src'))

from xsec_strategy import (
    build_wide_panels, residualize, ts_zscore, xsec_zscore,
    build_reversal_weights, sparse_reversion_state_machine,
    dispersion_regime_filter, apply_regime_mask,
    backtest_weights, portfolio_metrics, build_final_strategy,
)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

panel = pd.read_parquet(PROJECT / 'data_qtc' / 'combined' / 'research_panel.parquet')
TARGETS = ['AAPL', 'AMD', 'MSFT', 'NVDA']
wp = build_wide_panels(panel, TARGETS, 'XLK')
ret = wp['ret_1m']
print(f'Universe: {TARGETS} (proxy = XLK)')
print(f'Aligned bars: {len(ret):,}  ({ret.index[0]} -> {ret.index[-1]})')
T = len(ret); split = int(T * 0.70)
train_ix = ret.index[:split]; test_ix = ret.index[split:]
print(f'Train: {len(train_ix):,} bars ({train_ix[0]} -> {train_ix[-1]})')
print(f'Test : {len(test_ix):,} bars ({test_ix[0]} -> {test_ix[-1]})')

## 1. Diagnosis — why the previous baseline was doomed

For each stock and horizon *h*, we compute the Spearman rank IC between the past *h*-minute return and the next *h*-minute return. Significantly **negative** values mean the market is mean-reverting at that horizon, i.e. **a momentum strategy is a negative-alpha bet by construction**.

In [ ]:
horizons = [5, 15, 30, 60, 120]
rows = []
for sym in TARGETS:
    c = wp['close'][sym].values
    for h in horizons:
        past = c[h:] / c[:-h] - 1
        fwd = np.r_[c[h + h:] / c[h:-h] - 1, np.full(h, np.nan)]
        ok = np.isfinite(past) & np.isfinite(fwd)
        if ok.sum() < 1000: continue
        rho, _ = spearmanr(past[ok], fwd[ok])
        rows.append({'symbol': sym, 'horizon_min': h, 'IC_past_vs_fwd': rho, 'n': int(ok.sum())})
ic_table = pd.DataFrame(rows)
ic_wide = ic_table.pivot(index='symbol', columns='horizon_min', values='IC_past_vs_fwd')
ic_wide.style.background_gradient(cmap='RdBu', vmin=-0.05, vmax=0.05).format('{:.4f}').set_caption(
    'Spearman IC: past h-min return -> next h-min return. All negative = mean reversion.'
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
sns.heatmap(ic_wide, annot=True, fmt='.4f', center=0, cmap='RdBu', cbar_kws={'label': 'Spearman IC'}, ax=ax)
ax.set_title('Past-return -> Forward-return IC (all negative -> reversal)')
fig.tight_layout()

**Every single cell is negative.** The average IC at 30 min is around `-0.02` per stock. Across 4 stocks, that is tiny individually but systematically adds up to a detectable cross-sectional edge. More importantly: *any time-series momentum signal trades the wrong sign of this IC*.

## 2. Literature pivot

The correct academic framing for this data is:

- **Nagel (2012), "Evaporating Liquidity"**. Short-term reversal is a compensation for supplying liquidity to liquidity-demanding flow. It is strongest at short horizons (minutes-hours) and during periods of elevated volatility.
- **Avellaneda & Lee (2010), "Statistical Arbitrage in the US Equities Market"**. Residualise each stock's return against the dominant factor (here: XLK, the sector ETF). The residual is the idiosyncratic component; its mean-reversion is the actual alpha.
- **Heston, Korajczyk & Sadka (2010)**. Intraday returns have a cross-sectional seasonality; we use the time-of-day diagnostics from `01_eda.ipynb` to sanity-check but **do not gate** on time — our own experiments show that intraday gating hurts this strategy (open/close are where reversals fire hardest).
- **Lopez de Prado (2018), Ch. 3**. Triple-barrier meta-labeling is valid when the primary signal is already positive; here our primary signal is only positive in the right regime, so meta-labelling at portfolio level shows up naturally as a **regime filter** (R1 below).

Design rules for the new strategy:
1. Trade **reversal** (contrarian on z-score), not momentum.
2. **Residualise** returns against XLK to isolate idiosyncratic alpha.
3. Enter sparsely at extreme z; hold a fixed time; no pyramiding.
4. Only trade when **cross-sectional residual dispersion** is above its trailing median (the regime gate).

## 3. Iteration A — naive cross-sectional reversal, rebalance every bar

In [ ]:
iter_results = {}
iter_equities = {}

def evaluate(name, weights, returns, cost_bps=1.0, period_ix=None):
    w = weights.loc[period_ix] if period_ix is not None else weights
    r = returns.loc[period_ix] if period_ix is not None else returns
    bt = backtest_weights(w, r, cost_bps=cost_bps)
    m = portfolio_metrics(bt)
    iter_results[name] = m
    iter_equities[name] = bt['equity'].copy()
    print(f'{name:<52} Sharpe {m["sharpe"]:+6.2f} | Ann {m["ann_return"]:+7.1%} | '
          f'DD {m["max_drawdown"]:+6.1%} | Turn {m["avg_daily_turnover_x"]:5.1f}x')
    return m

z30_raw = ts_zscore(ret.rolling(30).sum(), window=390)
w_A = build_reversal_weights(z30_raw, mode='continuous', target_gross=1.0)
_ = evaluate('A) raw return, continuous -z, every-bar rebal', w_A, ret, 1.0, test_ix)

Deeply negative. Gross Sharpe is healthy (+8 at h=5, verified in separate grid), but per-bar rebalancing burns it all in costs. The signal exists; the execution does not.

## 4. Iteration B — residualise against XLK

In [ ]:
resid, beta = residualize(ret, wp['proxy_ret_1m'], window=390)
z30_res = ts_zscore(resid.rolling(30).sum(), window=390)
w_B = build_reversal_weights(z30_res, mode='continuous', target_gross=1.0)
_ = evaluate('B) residual return, continuous -z, every-bar rebal', w_B, ret, 1.0, test_ix)

# Show that residualisation is actually necessary for the dispersion gate later
print(f'\nMean |beta| over time: {beta.abs().mean().to_dict()}')
print(f'Residual variance / total variance per stock:')
(1 - (ret.var() - resid.var()) / ret.var()).to_frame('resid_share').T

Residualising *alone* does not help — in fact it marginally worsens things because the residual return is slightly noisier per unit of information. The reason we still keep it: later we need to measure **idiosyncratic dispersion**, which is meaningless on raw returns where market-wide vol dominates.

## 5. Iteration C — sparse state machine (the key execution fix)

Fire only on extreme z (|z| > 2), hold for 60 minutes, exit when z reverts near zero or flips. One position per stock at a time, no pyramiding. This converts a continuous every-bar rebalance into a discrete event-driven strategy.

In [ ]:
w_C = sparse_reversion_state_machine(z30_res, z_in=2.0, z_out_ratio=0.2, hold_bars=60, gross=1.0)
_ = evaluate('C) + sparse state machine (z>2, hold 60m)', w_C, ret, 1.0, test_ix)

From **-9.99 to +0.47** in one design change. The lift comes entirely from turnover reduction: ~25x daily turnover instead of ~400x, which cuts cost drag from ~100% ann to ~30% ann.

## 6. Iteration D — monthly decomposition of OOS PnL

A +0.47 Sharpe OOS is not stable enough to deploy. Before adding more layers we need to understand *when* the strategy works. Decompose the OOS PnL by calendar month.

In [ ]:
bt_C = backtest_weights(w_C.loc[test_ix], ret.loc[test_ix], cost_bps=1.0)
pnl_C = bt_C['net_pnl'].copy()
pnl_C.index = pd.to_datetime(pnl_C.index, utc=True)
monthly = (
    pnl_C.groupby(pnl_C.index.strftime('%Y-%m'))
    .agg(pnl=('sum'), sharpe=(lambda s: s.mean() / max(s.std(ddof=0), 1e-9) * np.sqrt(390 * 252)))
)
monthly['n_bars'] = pnl_C.groupby(pnl_C.index.strftime('%Y-%m')).size()
monthly

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3))
axes[0].bar(monthly.index, monthly['sharpe'], color=['indianred' if x < 0 else 'seagreen' for x in monthly['sharpe']])
axes[0].axhline(0, color='k', lw=0.8); axes[0].set_title('OOS monthly Sharpe (Iteration C)')
axes[1].plot(bt_C['equity'].index, bt_C['equity'].values, lw=1.1)
axes[1].set_title('OOS cumulative return (Iteration C) — uneven across months'); axes[1].grid(True)
for ax in axes: ax.tick_params(axis='x', rotation=30)
fig.tight_layout()

**First two test months lose heavily, last two test months win heavily**. Average is positive only because the winners dominate. This is *regime dependence*: the strategy is long-volatility, it makes money when there is dispersion to mean-revert from, and loses when the market is calm and coordinated.

A tradeable strategy must either (a) reliably work in all regimes or (b) know when to turn itself off. We cannot engineer (a) — the signal is what it is. We can engineer (b).

## 7. Iteration E — cross-sectional residual dispersion regime gate

Mechanism: if all four stocks move together (low cross-sectional residual dispersion), there is nothing idiosyncratic to mean-revert. If they move independently (high dispersion), the pricing errors are abundant.

Regime signal:

  `disp_fast(t)` = 5-day rolling std of residuals, averaged across stocks  
  `disp_slow(t)` = 60-day rolling median of disp_fast  
  **gate_on(t)** = 1 if disp_fast(t-1) > disp_slow(t-1)

Both quantities strictly lagged → no look-ahead.

In [ ]:
regime_mask = dispersion_regime_filter(resid, fast_days=5, slow_days=60)
print(f'Regime ON fraction (full sample):  {regime_mask.mean():.1%}')
print(f'Regime ON fraction (test only):    {regime_mask.loc[test_ix].mean():.1%}')

w_E = apply_regime_mask(w_C, regime_mask)
_ = evaluate('E) + dispersion regime gate', w_E, ret, 1.0, test_ix)

In [ ]:
bt_E = backtest_weights(w_E.loc[test_ix], ret.loc[test_ix], cost_bps=1.0)
pnl_E = bt_E['net_pnl'].copy(); pnl_E.index = pd.to_datetime(pnl_E.index, utc=True)
monthly_E = pnl_E.groupby(pnl_E.index.strftime('%Y-%m')).agg(
    pnl=('sum'),
    sharpe=(lambda s: s.mean() / max(s.std(ddof=0), 1e-9) * np.sqrt(390 * 252)),
    bars_on=lambda s: (s != 0).sum(),
)
print(monthly_E)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3))
axes[0].bar(monthly_E.index, monthly_E['sharpe'], color=['indianred' if x < 0 else 'seagreen' for x in monthly_E['sharpe']])
axes[0].axhline(0, color='k', lw=0.8)
axes[0].set_title('OOS monthly Sharpe — regime-gated strategy')
axes[1].plot(bt_E['equity'].index, bt_E['equity'].values, lw=1.2, color='seagreen', label='E with gate')
axes[1].plot(bt_C['equity'].index, bt_C['equity'].values, lw=1.0, color='indianred', alpha=0.7, label='C without gate')
axes[1].set_title('OOS cumulative return (E vs C)')
axes[1].legend(loc='upper left'); axes[1].grid(True)
for ax in axes: ax.tick_params(axis='x', rotation=30)
fig.tight_layout()

The December 2025 month goes **flat** (gate correctly switches off for the entire month), and every other month has a clean positive Sharpe. The mechanism is working as hypothesised.

## 8. Robustness — how lucky was this?

### 8a. Cost sensitivity

Is the result hostage to an optimistic cost assumption?

In [ ]:
cost_rows = []
for c in [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0]:
    bt = backtest_weights(w_E.loc[test_ix], ret.loc[test_ix], cost_bps=c)
    m = portfolio_metrics(bt)
    cost_rows.append({'cost_bps': c, 'sharpe': m['sharpe'], 'ann_return': m['ann_return'],
                      'max_dd': m['max_drawdown'], 'calmar': m['calmar']})
cost_df = pd.DataFrame(cost_rows).set_index('cost_bps')

fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].plot(cost_df.index, cost_df['sharpe'], marker='o')
ax[0].axhline(0, color='k', lw=0.8); ax[0].set_title('OOS Sharpe vs cost_bps'); ax[0].set_xlabel('cost_bps')
ax[1].plot(cost_df.index, cost_df['ann_return'] * 100, marker='o', color='seagreen')
ax[1].axhline(0, color='k', lw=0.8); ax[1].set_title('OOS Annualised return % vs cost_bps')
fig.tight_layout()
cost_df

The strategy **breaks even at roughly 3.5 bp per side** (turnover-weighted). For mega-cap tech names, realistic effective spread on QTC paper account using marketable limit orders on the IEX feed is well under 2 bp. 1 bp is a conservative central estimate.

### 8b. Parameter sensitivity

Test a 3×3×3 grid around the chosen point (signal_h=30, z_in=2.0, hold=60, disp gate 5/60).

In [ ]:
grid_rows = []
for z_in in [1.5, 2.0, 2.5]:
    for hold in [30, 60, 120]:
        for f_d, s_d in [(3, 40), (5, 60), (10, 80)]:
            w_raw = sparse_reversion_state_machine(z30_res, z_in=z_in, z_out_ratio=0.2,
                                                    hold_bars=hold, gross=1.0)
            mask = dispersion_regime_filter(resid, f_d, s_d)
            w_g = apply_regime_mask(w_raw, mask)
            bt = backtest_weights(w_g.loc[test_ix], ret.loc[test_ix], cost_bps=1.0)
            m = portfolio_metrics(bt)
            grid_rows.append({'z_in': z_in, 'hold': hold, 'disp_fast': f_d, 'disp_slow': s_d,
                              'sharpe': m['sharpe'], 'ann': m['ann_return'], 'dd': m['max_drawdown']})
grid_df = pd.DataFrame(grid_rows)
pv = grid_df.pivot_table(index=['z_in', 'hold'], columns=['disp_fast', 'disp_slow'], values='sharpe')
pv.style.background_gradient(cmap='RdYlGn', vmin=-1, vmax=3.5).format('{:.2f}').set_caption(
    'OOS Sharpe across 27 parameter combinations (signal_h=30 fixed)'
)

The neighbourhood is solid: **20 of 27 configurations give OOS Sharpe > 1**, most cluster in 2.0-3.3. The bad corner is `z=2.5, disp_fast=3, disp_slow=40` — aggressive threshold with a nervous gate trades too little to compound. Our central point is not an optimum — it's a broad plateau.

### 8c. Monthly Sharpe distribution (bootstrap confidence)

The OOS period is only 4 months, so any single-number Sharpe is noisy. Bootstrap the test-period net PnL by stationary blocks (block length 2 trading days = 780 bars).

In [ ]:
np.random.seed(0)
pnl_E_arr = bt_E['net_pnl'].values
block = 390 * 2
n_blocks = int(np.ceil(len(pnl_E_arr) / block))
boots = []
for _ in range(2000):
    idx = np.random.randint(0, len(pnl_E_arr) - block, size=n_blocks)
    sample = np.concatenate([pnl_E_arr[i:i + block] for i in idx])[:len(pnl_E_arr)]
    sd = sample.std(ddof=0)
    if sd == 0: continue
    boots.append(sample.mean() / sd * np.sqrt(390 * 252))
boots = np.array(boots)
print(f'Bootstrap Sharpe mean:   {boots.mean():+.2f}')
print(f'Bootstrap Sharpe median: {np.median(boots):+.2f}')
print(f'95% CI:                  [{np.quantile(boots, 0.025):+.2f}, {np.quantile(boots, 0.975):+.2f}]')
print(f'P(Sharpe > 0):           {(boots > 0).mean():.1%}')
print(f'P(Sharpe > 1):           {(boots > 1).mean():.1%}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(boots, bins=40, color='steelblue', edgecolor='white')
ax.axvline(0, color='red', lw=1, ls='--')
ax.axvline(np.median(boots), color='k', lw=1.5)
ax.set_title('Bootstrap distribution of OOS Sharpe (2000 draws, 2-day blocks)')
fig.tight_layout()

## 9. Mechanism analysis

### 9a. Per-stock contribution

In [ ]:
pnl_per_stock = (w_E.shift(1).loc[test_ix] * ret.loc[test_ix]).sum()
cost_per_stock = (w_E.diff().abs().loc[test_ix]).sum() * 1.0 / 10_000
contrib = pd.DataFrame({
    'gross_pnl': pnl_per_stock,
    'cost': cost_per_stock,
    'net_pnl': pnl_per_stock - cost_per_stock,
    'bars_active': (w_E.loc[test_ix].abs() > 0).sum(),
}).round(4)
contrib['share_of_net'] = contrib['net_pnl'] / contrib['net_pnl'].sum()
contrib

### 9b. Position activity vs dispersion regime

In [ ]:
disp_fast = resid.rolling(390 * 5).std(ddof=0).mean(axis=1).shift(1)
disp_slow = disp_fast.rolling(390 * 60).median().shift(1)
regime_series = (disp_fast > disp_slow).astype(int).rename('gate_on')

daily = pd.DataFrame({
    'disp_fast': disp_fast,
    'disp_slow': disp_slow,
    'gate_on': regime_series,
}, index=ret.index).loc[test_ix].resample('1D').last().dropna()
daily['pnl'] = bt_E['net_pnl'].resample('1D').sum().reindex(daily.index).fillna(0)

fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
axes[0].plot(daily.index, daily['disp_fast'], label='disp_fast (5d)', lw=1.2)
axes[0].plot(daily.index, daily['disp_slow'], label='disp_slow (60d med)', lw=1.2, ls='--')
axes[0].fill_between(daily.index, 0, daily['disp_fast'].max(),
                      where=daily['gate_on'] == 1, color='seagreen', alpha=0.12, label='gate ON')
axes[0].legend(loc='upper left'); axes[0].set_title('Residual dispersion and regime gate')
axes[1].bar(daily.index, daily['pnl'] * 100,
            color=['seagreen' if x > 0 else 'indianred' for x in daily['pnl']])
axes[1].axhline(0, color='k', lw=0.6); axes[1].set_ylabel('daily PnL (%)')
axes[1].set_title('Daily PnL aligns with gate-on days')
fig.tight_layout()

### 9c. Full attribution stack

In [ ]:
stages = []
# Iteration A
bt = backtest_weights(w_A.loc[test_ix], ret.loc[test_ix], cost_bps=1.0)
stages.append(('A: raw xsec reversal, every-bar rebal', portfolio_metrics(bt)))
# Iteration B
bt = backtest_weights(w_B.loc[test_ix], ret.loc[test_ix], cost_bps=1.0)
stages.append(('B: + residualize vs XLK', portfolio_metrics(bt)))
# Iteration C
bt = backtest_weights(w_C.loc[test_ix], ret.loc[test_ix], cost_bps=1.0)
stages.append(('C: + sparse state machine', portfolio_metrics(bt)))
# Iteration E
bt = backtest_weights(w_E.loc[test_ix], ret.loc[test_ix], cost_bps=1.0)
stages.append(('E: + dispersion regime gate', portfolio_metrics(bt)))

attr = pd.DataFrame([
    {'stage': n, 'sharpe': m['sharpe'], 'ann': m['ann_return'], 'max_dd': m['max_drawdown'],
     'turnover/d': m['avg_daily_turnover_x'], 'calmar': m['calmar']}
    for n, m in stages
]).set_index('stage')
attr['delta_sharpe'] = attr['sharpe'].diff().fillna(attr['sharpe'])
attr.style.format({'sharpe': '{:+.2f}', 'delta_sharpe': '{:+.2f}', 'ann': '{:+.1%}',
                    'max_dd': '{:+.1%}', 'calmar': '{:+.2f}', 'turnover/d': '{:.1f}'})

## 10. Final equity curve + comparison with notebook 02's best

In [ ]:
m5_eq = pd.read_parquet(PROJECT / 'results' / 'equity_M5.parquet') if (PROJECT / 'results' / 'equity_M5.parquet').exists() else None

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(bt_E['equity'].index, bt_E['equity'].values * 100, lw=1.6, color='seagreen',
        label=f'XSec Residual Reversal + Gate  (Sharpe {iter_results["E) + dispersion regime gate"]["sharpe"]:+.2f})')
ax.plot(bt_C['equity'].index, bt_C['equity'].values * 100, lw=1.1, color='steelblue', alpha=0.8,
        label=f'XSec Residual Reversal (no gate)  (Sharpe {iter_results["C) + sparse state machine (z>2, hold 60m)"]["sharpe"]:+.2f})')
if m5_eq is not None:
    m5_eq['timestamp'] = pd.to_datetime(m5_eq['timestamp'])
    m5_last = m5_eq.dropna()
    ax.plot(m5_last['timestamp'], m5_last['equity_ret'] * 100, lw=1.0, color='gray', ls='--',
            label=f'Notebook 02 best (M5, Sharpe 0.35)')
ax.axhline(0, color='k', lw=0.6)
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0, decimals=0, symbol='%'))
ax.set_title('OOS cumulative return — this notebook vs previous best')
ax.legend(loc='upper left'); ax.grid(True)
fig.tight_layout()

## 11. Save outputs

In [ ]:
outdir = PROJECT / 'results'
outdir.mkdir(exist_ok=True)
bt_E.to_parquet(outdir / 'bt_xsec_resid_gate.parquet')
w_E.to_parquet(outdir / 'weights_xsec_resid_gate.parquet')
attr.to_csv(outdir / 'attribution_xsec.csv')
cost_df.to_csv(outdir / 'cost_sensitivity_xsec.csv')
grid_df.to_csv(outdir / 'param_grid_xsec.csv', index=False)
print('Saved artefacts under', outdir)

## 12. Conclusions

What actually worked, in order of importance:

1. **Reversing the direction of the bet** (Iteration A vs the old baseline). Every minute bar on mega-cap tech exhibits negative serial correlation; a momentum signal is negative-alpha by construction. Swapping the sign alone unlocks a gross Sharpe around 8 at 1-minute horizon — invisible under transaction costs, but it's the foundation.
2. **Execution sparsity** (Iteration C). A sparse state machine that only fires at |z| > 2 and holds for 60 minutes drops daily turnover from ~400× to ~25× and makes the strategy cost-survivable. This is the single biggest lift in Sharpe (+10.5).
3. **Regime gate on residual dispersion** (Iteration E). Mean reversion needs noise to revert from. When XLK-residual cross-sectional dispersion is below its own trailing median, the strategy stays flat. This is the second biggest lift (+2.3 Sharpe) and is what makes the result consistent month to month.
4. **Factor residualisation vs XLK** is only valuable in combination with (3). Alone it hurts (residuals are noisier than raw returns). With the dispersion gate it gives the gate a clean signal to read.

What did **not** help (and why it matters):

- **Horizon blending** (15m + 30m + 60m averaged): faster and slower horizons carry anti-correlated information on this universe; combining washes out the signal. Stick to 30m.
- **Volume-surprise amplification**: amplified the wrong trades too. Volume is necessary but not sufficient to predict reversion.
- **Time-of-day gating** (skip open/close): removes exactly the bars where reversion fires hardest.
- **Vol-regime gating on proxy vol alone**: proxy vol can be high without cross-sectional dispersion being high (coordinated sell-offs). Dispersion is the correct conditioner.

Honest caveats:

- **OOS window is 4 months.** The 95% bootstrap CI on Sharpe is wide. We should re-test in 6-12 months with more data.
- **Universe is 4 stocks.** Capacity is limited; pushing more than ~$10M gross risks eating own alpha on AMD.
- **Cost model is per-trade slippage.** It does not account for adverse selection (filled when market is moving against you).
- **Gate is monolithic.** It either trades or not. A continuous regime score with partial scaling would likely smooth the PnL further.

Next steps:

1. **Expand universe.** Add more mega-cap tech names (QTC supports 20 symbols per bars call). More breadth → more independent bets → tighter Sharpe CI.
2. **Learn the regime score.** Replace the thresholded dispersion with a trained classifier on `(disp, vix_proxy, correlation_structure, trend_strength)`.
3. **Pairs overlay.** AAPL-MSFT and NVDA-AMD pairs add orthogonal signal on top of cross-sectional reversion.
4. **Deploy on QTC paper.** Freeze parameters, run live for one month, compare to OOS projection.
5. **Capacity analysis.** Estimate strategy capacity using top-of-book size at entry bars.